In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plot_style import (
    FONT_SIZES, LARGE_FIG_SIZE, SMALL_FIG_SIZE, ALPHA,
    apply_style, save_fig_both,
)

In [ ]:
apply_style()

In [ ]:
pdb_df = pd.read_csv('science.abo7201_data_s4.csv')
pdb_df = pdb_df.transpose()
pdb_df.columns = pdb_df.iloc[0].tolist()
pdb_df.drop(index="Dataset", inplace=True)
pdb_df['lig_bfactor'] = pd.to_numeric(pdb_df['B-factor for ligands'], errors='coerce')
pdb_df['structure_name'] = pdb_df.index
pdb_df = pdb_df[pdb_df['structure_name'].apply(lambda x: 'Mpro-P' in x or 'Mpro-x' in x)]
pdb_df['resolution'] = pdb_df['Resolution range'].apply(lambda x: float(x.split('-')[1].split('(')[0].strip()))
pdb_df['rfactor'] = pdb_df['R-factor'].apply(lambda x: float(x.split('(')[0].strip()))
pdb_df['rfree'] = pdb_df['R-free'].apply(lambda x: float(x.split('(')[0].strip()))

In [ ]:
sns.histplot(pdb_df, x='lig_bfactor', bins=25)
plt.title('Distribution of B-factors for Ligands')
plt.xlabel('B-factor for Ligands')
plt.ylabel('Frequency')
plt.savefig('ligand_bfactor_distribution.png', dpi=300)

In [ ]:
g = sns.PairGrid(pdb_df[['lig_bfactor', 'resolution', 'rfactor', 'rfree']].dropna())
g.map_upper(sns.scatterplot, alpha=1)
g.map_lower(sns.scatterplot, alpha=1)
g.map_diag(sns.histplot, bins=20)
plt.suptitle('Pairwise Comparisons of Structure Quality Metrics', y=1.02)
plt.savefig('pairwise_quality_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from scipy import stats

def annotate_corr(x, y, **kwargs):
    mask = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[mask], y[mask]
    r, p = stats.pearsonr(x, y)
    r2 = r**2
    ax = plt.gca()
    ax.annotate(f'ρ = {r:.2f}\nR² = {r2:.2f}',
                xy=(0.05, 0.85), xycoords='axes fraction',
                fontsize=10)

import numpy as np
g = sns.PairGrid(pdb_df[['lig_bfactor', 'resolution', 'rfactor', 'rfree']].dropna())
g.map_upper(sns.scatterplot, alpha=1)
g.map_upper(annotate_corr)
g.map_lower(sns.scatterplot, alpha=1)
g.map_diag(sns.histplot, bins=20)
plt.suptitle('Pairwise Comparisons of Structure Quality Metrics', y=1.02)
plt.savefig('pairwise_quality_metrics.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
pdb_df.columns

# Add lig b factor to docking data

In [ ]:
from harbor.analysis.cross_docking import DockingDataModel, DataFrameModel, DataFrameType

In [ ]:
original_data = DockingDataModel.deserialize("/Users/apayne/Downloads/full_cross_dock_v2/combined_docking_results/ALL_1_poses.json")

In [ ]:
models = original_data.to_models()

In [ ]:
type(models)

In [ ]:
type(models[0])

In [ ]:
real_structure_name = {}
for structure in pdb_df.structure_name:
    for reference in original_data.dataframe.Reference_Structure.unique():
        if structure in reference:
            real_structure_name[structure] = reference
            break

In [ ]:
len(real_structure_name)

In [ ]:
len(original_data.dataframe.Reference_Structure.unique())

In [ ]:
'Mpro-x3325' in pdb_df.structure_name

In [ ]:
pdb_df.structure_name

In [ ]:
set(original_data.dataframe.Reference_Structure.unique()) - set(real_structure_name.values())

In [ ]:
set(pdb_df.structure_name) - set(real_structure_name.keys())

In [ ]:
real_structure_name

In [ ]:
simplified_df = pdb_df[['Compound', 'B-factor for ligands']]
simplified_df = simplified_df.reset_index()
simplified_df.columns = ['Reference_Structure', 'Reference_Ligand', 'Ligand_B_Factor']
simplified_df["Reference_Structure"] = simplified_df["Reference_Structure"].apply(lambda x: real_structure_name.get(x, None))
simplified_df["Ligand_B_Factor"] = simplified_df["Ligand_B_Factor"].astype(float)
query_lig_b_factor_data = DataFrameModel(name='RefCrystalData', 
                                         dataframe=simplified_df,
                                         type=DataFrameType.REFERENCE,
                                         key_columns=["Reference_Structure"],
                                         value_columns=["Ligand_B_Factor"])

In [ ]:
models = original_data.to_models()
models.append(query_lig_b_factor_data)

In [ ]:
updated = DockingDataModel.from_models(models)

In [ ]:
crystal_data_only = updated.dataframe.dropna()

In [ ]:
crystal_data_only.nunique()

In [ ]:
success_mean = (
  crystal_data_only.groupby("Reference_Structure")["PoseData_RMSD"]
  .apply(lambda x: (x < 2.0).mean())
  .reset_index(name="success_rate_mean")
)

In [ ]:
merged = success_mean.merge(simplified_df, left_on="Reference_Structure", right_on="Reference_Structure", how='inner')
merged["bfactor_bin"] = pd.qcut(merged["Ligand_B_Factor"], q=4, labels=["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"])

In [ ]:
merged

In [ ]:
sns.scatterplot(merged, x = "Ligand_B_Factor", y = "success_rate_mean")

In [ ]:
sns.scatterplot(merged, x = "bfactor_bin", y = "success_rate_mean")

# Success Rate vs B factor by Quartile

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=42)

def bootstrap_bin_ci(values, n_bootstraps=1000, ci=0.95):
    replicates = np.array([
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_bootstraps)
    ])
    lo = np.percentile(replicates, (1 - ci) / 2 * 100)
    hi = np.percentile(replicates, (1 + ci) / 2 * 100)
    return lo, hi

bin_stats = []
for bin_label, group in merged.groupby("bfactor_bin", observed=True):
    values = group["success_rate_mean"].dropna().values
    mean = values.mean()
    lo, hi = bootstrap_bin_ci(values)
    bin_stats.append({"bfactor_bin": bin_label, "mean": mean, "ci_low": lo, "ci_high": hi, "n": len(values)})

bin_stats_df = pd.DataFrame(bin_stats)

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)

sns.stripplot(data=merged.dropna(subset=["bfactor_bin", "success_rate_mean"]),
              x="bfactor_bin", y="success_rate_mean",
              ax=ax, alpha=0.3, color="gray", size=4, jitter=True, zorder=1)

x = range(len(bin_stats_df))
ax.errorbar(
    x=x, y=bin_stats_df["mean"],
    yerr=[bin_stats_df["mean"] - bin_stats_df["ci_low"],
          bin_stats_df["ci_high"] - bin_stats_df["mean"]],
    fmt="o", color="steelblue", capsize=5, lw=2, ms=8, zorder=2,
)
ax.set_xticks(list(x))
ax.set_xticklabels(bin_stats_df["bfactor_bin"].tolist(), fontsize=FONT_SIZES["ticks"])
ax.tick_params(axis="y", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("Ligand B-factor quartile", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Success rate (RMSD < 2 Å)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
for i, row in bin_stats_df.iterrows():
    ax.text(i, row["ci_high"] + 0.01, f"n={row['n']}", ha="center", fontsize=FONT_SIZES["ticks"])
sns.despine()
plt.tight_layout()
save_fig_both(fig, "success_rate_vs_bfactor_quartile")

# Plot b factor vs date colored by quartile

In [ ]:
date_info = crystal_data_only.groupby("Reference_Structure").head(1)
date_info["Deposition Date"] = date_info["RefData_Date"].apply(lambda x: x[10])

In [ ]:
date_info

In [ ]:
sns.scatterplot(crystal_data_only.groupby("Reference_Structure").head(1), x = "RefData_Date", y="RefCrystalData_Ligand_B_Factor")

In [ ]:
# one row per reference structure: date + B-factor
ref_dates = (
  crystal_data_only[["Reference_Structure", "RefData_Date", "RefCrystalData_Ligand_B_Factor"]]
  .drop_duplicates("Reference_Structure")
  .rename(columns={"RefCrystalData_Ligand_B_Factor": "Ligand_B_Factor"})
)
ref_dates["RefData_Date"] = pd.to_datetime(ref_dates["RefData_Date"])

fig, ax = plt.subplots(figsize=(8, 4))

sc = ax.scatter(
  ref_dates["RefData_Date"],
  ref_dates["Ligand_B_Factor"],
  c=ref_dates["Ligand_B_Factor"],
  cmap="plasma",
  alpha=0.7,
  s=25,
  edgecolors="none",
)
plt.colorbar(sc, ax=ax, label="Ligand B-factor (Ų)")
ax.set_xlabel("Crystal structure date")
ax.set_ylabel("Ligand B-factor (Ų)")
ax.set_title("Ligand B-factor over time")
fig.autofmt_xdate()
sns.despine()
plt.tight_layout()

In [ ]:
# ref_dates = (
#   crystal_data_only[["Reference_Structure", "RefData_Date", "RefCrystalData_Ligand_B_Factor"]]
#   .drop_duplicates("Reference_Structure")
#   .rename(columns={"RefCrystalData_Ligand_B_Factor": "Ligand_B_Factor"})
# )
# ref_dates["RefData_Date"] = pd.to_datetime(ref_dates["RefData_Date"])
ref_dates_merged = ref_dates.merge(
    merged[["Reference_Structure", "bfactor_bin"]],
    on="Reference_Structure", how="left"
)

palette = dict(zip(
    ["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"],
    sns.color_palette("plasma", 4)
))

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)
for bin_label, group in ref_dates_merged.groupby("bfactor_bin", observed=True):
    ax.scatter(
        group["RefData_Date"], group["Ligand_B_Factor"],
        color=palette[bin_label], label=bin_label.replace("\n", " "),
        alpha=0.7, s=25, edgecolors="none",
    )

ax.tick_params(axis="both", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("Crystal structure date", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Ligand B-factor (Ų)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
# legend = ax.legend(title="B-factor quartile", bbox_to_anchor=(1.01, 1), loc="lower right")
legend = ax.legend(title="B-factor quartile")
plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
fig.autofmt_xdate()
sns.despine()
plt.tight_layout()
save_fig_both(fig, "bfactor_over_time_by_quartile")

# Ligand b factor colored by series

In [ ]:
ref_dates_merged["series"] = ref_dates_merged["Reference_Structure"].apply(
    lambda x: "Monoclinic (x-series)" if "Mpro-x" in x else "Orthorhombic (p-series)"
)

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)
for series, group in ref_dates_merged.groupby("series"):
    ax.scatter(
        group["RefData_Date"], group["Ligand_B_Factor"],
        label=series, alpha=0.7, s=25, edgecolors="none",
    )

ax.tick_params(axis="both", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("Crystal structure date", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Ligand B-factor (Ų)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
legend = ax.legend(title="Series")
plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
fig.autofmt_xdate()
sns.despine()
plt.tight_layout()
save_fig_both(fig, "bfactor_over_time_by_series")

In [ ]:
# Success rate vs B-factor quartile, split by crystal series
# ref_dates_merged already has "series" column from the cell above
merged_series = merged.merge(
  ref_dates_merged[["Reference_Structure", "series"]].drop_duplicates(),
  on="Reference_Structure", how="left"
)

series_list = sorted(merged_series["series"].dropna().unique())
fig, axes = plt.subplots(1, len(series_list), figsize=LARGE_FIG_SIZE, sharey=True)

for ax, series_name in zip(axes, series_list):
  sub = merged_series[merged_series["series"] == series_name].dropna(subset=["bfactor_bin", "success_rate_mean"])

  # Compute bootstrap CI per quartile bin within this series
  bin_stats_s = []
  for bin_label, group in sub.groupby("bfactor_bin", observed=True):
      values = group["success_rate_mean"].dropna().values
      mean = values.mean()
      lo, hi = bootstrap_bin_ci(values)
      bin_stats_s.append({"bfactor_bin": bin_label, "mean": mean, "ci_low": lo, "ci_high": hi, "n": len(values)})
  bin_stats_s_df = pd.DataFrame(bin_stats_s)

  # Individual structure success rates as gray points
  sns.stripplot(data=sub, x="bfactor_bin", y="success_rate_mean",
                ax=ax, alpha=0.3, color="gray", size=4, jitter=True, zorder=1)

  # Bootstrap mean ± 95% CI overlay
  x = range(len(bin_stats_s_df))
  ax.errorbar(
      x=x, y=bin_stats_s_df["mean"],
      yerr=[bin_stats_s_df["mean"] - bin_stats_s_df["ci_low"],
            bin_stats_s_df["ci_high"] - bin_stats_s_df["mean"]],
      fmt="o", color="steelblue", capsize=5, lw=2, ms=8, zorder=2,
  )
  ax.set_xticks(list(x))
  ax.set_xticklabels(bin_stats_s_df["bfactor_bin"].tolist(), fontsize=FONT_SIZES["ticks"])
  ax.tick_params(axis="y", labelsize=FONT_SIZES["ticks"])
  ax.set_xlabel("Ligand B-factor quartile", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
  ax.set_title(series_name, fontsize=FONT_SIZES["xlabel"], fontweight="bold")
  # Annotate n per bin above the error bar cap
  for i, row in bin_stats_s_df.iterrows():
      ax.text(i, row["ci_high"] + 0.01, f"n={row['n']}", ha="center", fontsize=FONT_SIZES["ticks"])

# Only left panel needs y-label since axes share y scale
axes[0].set_ylabel("Success rate (RMSD < 2 Å)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
axes[1].set_ylabel("")
sns.despine()
plt.tight_layout()
save_fig_both(fig, "success_rate_vs_bfactor_quartile_by_series")

# success by quartile split by series

In [ ]:
# Single panel: both series within each B-factor quartile, offset for visual separation
# Uses manual scatter+jitter so errorbar x positions align exactly with the point clouds
merged_series = merged.merge(
  ref_dates_merged[["Reference_Structure", "series"]].drop_duplicates(),
  on="Reference_Structure", how="left"
)

series_list = sorted(merged_series["series"].dropna().unique())
palette = dict(zip(series_list, sns.color_palette(n_colors=len(series_list))))
bin_labels = ["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"]
DODGE = 0.2
JITTER = 0.05
rng_j = np.random.default_rng(seed=0)

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)

offsets = [-DODGE / 2, DODGE / 2]

for series_name, x_offset in zip(series_list, offsets):
  sub = merged_series[merged_series["series"] == series_name].dropna(
      subset=["bfactor_bin", "success_rate_mean"]
  )

  # Bootstrap CI per quartile bin; reindex to all 4 bins so empty ones show n=0
  bin_stats_s = []
  for bin_label, group in sub.groupby("bfactor_bin", observed=True):
      values = group["success_rate_mean"].dropna().values
      mean = values.mean()
      lo, hi = bootstrap_bin_ci(values)
      bin_stats_s.append({"bfactor_bin": bin_label, "mean": mean, "ci_low": lo, "ci_high": hi, "n": len(values)})
  bin_stats_s_df = (
      pd.DataFrame(bin_stats_s)
      .set_index("bfactor_bin")
      .reindex(bin_labels)  # ensures all 4 bins present, NaN where missing
      .reset_index()
      .assign(n=lambda d: d["n"].fillna(0).astype(int))
  )

  # Individual points with jitter
  x_cat = np.array([bin_labels.index(b) for b in sub["bfactor_bin"]])
  jitter = rng_j.uniform(-JITTER, JITTER, len(sub))
  ax.scatter(x_cat + x_offset + jitter, sub["success_rate_mean"].values,
             color=palette[series_name], alpha=0.3, s=16, edgecolors="none", zorder=1)

  # Mean ± 95% CI — skipped automatically for NaN rows by errorbar
  x_err = [bin_labels.index(b) + x_offset for b in bin_stats_s_df["bfactor_bin"]]
  ax.errorbar(
      x=x_err, y=bin_stats_s_df["mean"],
      yerr=[bin_stats_s_df["mean"] - bin_stats_s_df["ci_low"],
            bin_stats_s_df["ci_high"] - bin_stats_s_df["mean"]],
      fmt="o", color=palette[series_name], capsize=5, lw=2, ms=8, zorder=2,
      label=series_name,
  )
  # n annotation — always shown, including n=0 for empty bins
  for x_pos, row in zip(x_err, bin_stats_s_df.itertuples()):
      ax.text(x_pos, row.ci_high + 0.01 if pd.notna(row.ci_high) else 0.4,
              f"n={row.n}", ha="center", fontsize=FONT_SIZES["panel_label"])

ax.set_xticks(range(len(bin_labels)))
ax.set_xticklabels(bin_labels, fontsize=FONT_SIZES["ticks"])
ax.tick_params(axis="y", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("Ligand B-factor quartile", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Success rate (RMSD < 2 Å)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
legend = ax.legend(title="Series")
plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
sns.despine()
plt.tight_layout()
save_fig_both(fig, "success_rate_vs_bfactor_quartile_by_series")

# Make same plots for R-factor

In [ ]:
rfactor_lookup = (
    pdb_df[["structure_name", "rfactor"]]
    .assign(Reference_Structure=lambda d: d["structure_name"].map(real_structure_name))
    .dropna(subset=["Reference_Structure"])
)
rfactor_merged = merged.merge(rfactor_lookup[["Reference_Structure", "rfactor"]], on="Reference_Structure", how="left")
rfactor_merged["rfactor_bin"] = pd.qcut(rfactor_merged["rfactor"], q=4, labels=["Q1\n(lowest)", "Q2", "Q3", "Q4\n(highest)"])

bin_stats_rf = []
for bin_label, group in rfactor_merged.groupby("rfactor_bin", observed=True):
    values = group["success_rate_mean"].dropna().values
    mean = values.mean()
    lo, hi = bootstrap_bin_ci(values)
    bin_stats_rf.append({"rfactor_bin": bin_label, "mean": mean, "ci_low": lo, "ci_high": hi, "n": len(values)})
bin_stats_rf_df = pd.DataFrame(bin_stats_rf)

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)
sns.stripplot(data=rfactor_merged.dropna(subset=["rfactor_bin"]), x="rfactor_bin", y="success_rate_mean",
              ax=ax, alpha=0.3, color="gray", size=4, jitter=True, zorder=1)
x = range(len(bin_stats_rf_df))
ax.errorbar(x=x, y=bin_stats_rf_df["mean"],
            yerr=[bin_stats_rf_df["mean"] - bin_stats_rf_df["ci_low"],
                  bin_stats_rf_df["ci_high"] - bin_stats_rf_df["mean"]],
            fmt="o", color="steelblue", capsize=5, lw=2, ms=8, zorder=2)
ax.set_xticks(list(x))
ax.set_xticklabels(bin_stats_rf_df["rfactor_bin"].tolist(), fontsize=FONT_SIZES["ticks"])
ax.tick_params(axis="y", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("R-factor quartile", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("Success rate (RMSD < 2 Å)", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
for i, row in bin_stats_rf_df.iterrows():
    ax.text(i, row["ci_high"] + 0.01, f"n={row['n']}", ha="center", fontsize=FONT_SIZES["ticks"])
sns.despine()
plt.tight_layout()
save_fig_both(fig, "success_rate_vs_rfactor_quartile")

In [ ]:
ref_dates_rf = ref_dates_merged.merge(rfactor_lookup[["Reference_Structure", "rfactor"]], on="Reference_Structure", how="left")

fig, ax = plt.subplots(figsize=SMALL_FIG_SIZE)
for series, group in ref_dates_rf.dropna(subset=["rfactor"]).groupby("series"):
    ax.scatter(group["RefData_Date"], group["rfactor"],
               label=series, alpha=0.7, s=25, edgecolors="none")

ax.tick_params(axis="both", labelsize=FONT_SIZES["ticks"])
ax.set_xlabel("Crystal structure date", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
ax.set_ylabel("R-factor", fontsize=FONT_SIZES["ylabel"], fontweight="bold")
legend = ax.legend(title="Series")
plt.setp(legend.get_title(), fontsize=FONT_SIZES["legend_title"], fontweight="bold")
plt.setp(legend.get_texts(), fontsize=FONT_SIZES["legend_text"])
fig.autofmt_xdate()
sns.despine()
plt.tight_layout()
save_fig_both(fig, "rfactor_over_time_by_series")